In [ ]:
import os
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.tools import tool

os.environ["TAVILY_API_KEY"] = "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"

class FlightInput(BaseModel):
    departure: Optional[str] = Field(default=None, description="出发地")
    arrival: Optional[str] = Field(default=None, description="目的地")
    departure_time: Optional[str] = Field(default=None, description="出发时间")
    return_time: Optional[str] = Field(default=None, description="回程时间")
    adults: Optional[int] = Field(default=1, description="成人数量")
    children: Optional[int] = Field(default=0, description="儿童数量")
    infants_in_seat: Optional[int] = Field(default=0, description="占座婴儿数量")
    infants_without_seat: Optional[int] = Field(default=0, description="不占座婴儿数量")

    model_config = {
        "extra": "forbid"  # 禁止额外字段
    }

class FlightInputSchema(BaseModel):
    params: FlightInput
    
    model_config = {
        "extra": "forbid"  # 禁止额外字段
    }

@tool(args_schema=FlightInputSchema)
def Flight_finders(params: FlightInput) -> str:
    """查询航班信息，返回结构化搜索结果"""
    # 使用新版 TavilySearch
    from langchain_tavily import TavilySearch
    tavily_tool = TavilySearch(
        max_results=5,
        search_depth="advanced",
        language="zh",
        include_answer=True,
        use_cache=False
    )

    # 拼接搜索关键词
    search_parts = []
    if params.departure:
        search_parts.append(f"从{params.departure}")
    if params.arrival:
        search_parts.append(f"到{params.arrival}")
    
    search_parts.append("飞机航班 时刻表 票价")
    
    if params.departure_time:
        search_parts.append(f"出发时间{params.departure_time}")
    if params.return_time:
        search_parts.append(f"返程时间{params.return_time}")
    if params.adults and params.adults != 1:
        search_parts.append(f"成人{params.adults}人")
    if params.children and params.children > 0:
        search_parts.append(f"儿童{params.children}人")
    if params.infants_in_seat and params.infants_in_seat > 0:
        search_parts.append(f"占座婴儿{params.infants_in_seat}人")
    if params.infants_without_seat and params.infants_without_seat > 0:
        search_parts.append(f"不占座婴儿{params.infants_without_seat}人")

    search_query = " ".join(search_parts).strip()

    # 执行搜索
    try:
        result = tavily_tool.run(search_query)
        final_result = f"""
🔍 搜索关键词：{search_query}
====================================
📌 航班查询结果：
{result}
        """.strip()
        return final_result
    except Exception as e:
        return f"❌ 查询失败：{str(e)}"